In [ ]:
%matplotlib inline
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from scipy.integrate import quad
import matplotlib as mpl
from scipy.constants import c, e, m_e, epsilon_0
mpl.rcParams['animation.embed_limit'] = 50

In [ ]:


Q = 10**(-8)
q = -e
m = m_e
k = 1 / (4 * np.pi * epsilon_0)
R = 0.1
x0 = 0.0005
v0 = 0


def calculate_t_max(x_start, num_cycles=3):
    def U(x):
        return (k * q * Q) / np.sqrt(R**2 + x**2)
    
    E_total = U(x_start)
    
    def time_element(x):
        Kinetic_Energy = E_total - U(x)
        if Kinetic_Energy <= 0: 
            return 0 
        
        velocity = np.sqrt((2/m) * Kinetic_Energy)
        return 1.0 / velocity
        
    quarter_period, error = quad(time_element, 0, x_start * 0.9999)
    
    period = 4 * quarter_period
    return period * num_cycles

t_max = calculate_t_max(x0, num_cycles=3)
t_eval = np.linspace(0, t_max, 5000)



def diff_eq(t,S):
    x, x_dot = S
    dx_dt = x_dot
    dxdot_dt = (k * q * Q * x)/(m*((R**2 + x**2)**(3/2)))
    return [dx_dt, dxdot_dt]

sol = solve_ivp(diff_eq, [0, t_max], [x0, v0], t_eval=t_eval, method="Radau", rtol=1e-8, atol=1e-10)

x_vals = sol.y[0]
v_vals = sol.y[1]
t_vals = sol.t

fig, ((ax, ax2),(ax3,ax4)) = plt.subplots(2, 2, figsize=(12, 10))

w = np.sqrt((k * abs(q) * Q) / (m * R**3))
x_approx = x0*np.cos(w*t_vals)
v_approx = -x0*w*np.sin(w*t_vals)

ax.set_xlim(0, t_max)
ax.set_ylim(-x0 * 1.5, x0 * 1.5)
ax.set_xlabel(r'$t$ (s)')
ax.set_ylabel(r'$x$ (m)')
ax.plot(t_vals, x_vals, color = 'cornflowerblue', label = 'Non-relativistic Motion' )
ax.plot(t_vals, x_approx, color = 'darkorange', label = 'SHO Motion')
ax.text(0.05 * t_max, x0 * 1.2, r'$x_0 = $' + f'{x0} m', fontsize=12)
ax.legend()
point_true, = ax.plot([], [], 'bo', markersize=8)
point_approx, = ax.plot([], [], 'ro', markersize=8)

ax2.vlines(x=0, ymin=-R, ymax=R, color='black', linewidth=4, label= f"Charged Ring ($Q$ = {Q} C)")
ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlim(-x0 *1.5, x0 *1.5)
ax2.set_ylim(-R * 1.5, R * 1.5) 
ax2.set_xlabel(r'$x$ (m)')
ax2.set_ylabel(r'$y$ (m)')
particle, = ax2.plot([], [], 'bo', markersize=8, label = 'Non-relativistic motion')
particle_approx, = ax2.plot([], [], 'ro', markersize=8, label = 'SHO motion')
ax2.legend()

U_taylor = (1/R)*k*q*Q - (1 / (2 * R**3))*k*q*Q*(x_vals)**2

K_vals_c = (1/2) * m * (sol.y[1])**2
U_vals_c = k * q * Q / ((R**2)+(x_vals)**2)**0.5
E_tot_c = U_vals_c + K_vals_c

ax3.plot(x_vals, U_vals_c, color = 'grey', label = 'Potential Energy')
ax3.plot(x_vals, U_taylor, color = 'darkorange', label = 'SHO Potential')
ax3.set_xlabel(r'$x$ (m)')
ax3.set_ylabel(r'$U$ (J)')
ax3.legend()
point_potential, = ax3.plot([], [], 'ko', markersize=8)
point_potential_SHO, = ax3.plot([], [], 'ro', markersize=8)

ax4.plot(x_vals, v_vals, color = 'cornflowerblue', label = 'Non-relativistic Phase Plot')
ax4.plot(x_approx, v_approx, color = 'darkorange', label = 'SHO Phase Plot')
point_phase_t, = ax4.plot([], [], 'bo', markersize=8)
point_phase_a, = ax4.plot([], [], 'ro', markersize=8)

ax4.set_xlabel(r'$x$ (m)')
ax4.set_ylabel(r'$\frac{dx}{dt}$ (m/s)')
ax4.legend()

for axis in [ax, ax2, ax3, ax4]:
    axis.ticklabel_format(style='sci', axis='both', scilimits=(-2, 2))

def init():
    point_true.set_data([], [])
    point_approx.set_data([], [])
    particle.set_data([], [])
    particle_approx.set_data([], [])
    point_potential.set_data([], [])
    point_potential_SHO.set_data([], [])
    point_phase_t.set_data([], [])
    point_phase_a.set_data([], [])
    return (point_true, point_approx, particle, particle_approx, point_potential, point_potential_SHO, point_phase_t, point_phase_a)

def update(frame):
    point_true.set_data([t_vals[frame]], [x_vals[frame]])
    point_approx.set_data([t_vals[frame]], [x_approx[frame]])
    particle.set_data([x_vals[frame]], [0])
    particle_approx.set_data([x_approx[frame]], [0])
    point_potential.set_data([x_vals[frame]], [U_vals_c[frame]]) 
    point_potential_SHO.set_data([x_vals[frame]], [U_taylor[frame]])
    point_phase_t.set_data([x_vals[frame]], [v_vals[frame]])
    point_phase_a.set_data([x_approx[frame]], [v_approx[frame]])
    
    return (point_true, point_approx, particle, particle_approx, point_potential, point_potential_SHO, point_phase_t, point_phase_a)

step_size = 12 

ani = FuncAnimation(fig, update, frames=range(0, len(t_eval), step_size),
                    init_func=init, blit=True, interval=25)

plt.tight_layout()
plt.close()

U_start = (k * q * Q) / np.sqrt(R**2 + x0**2)
U_center = (k * q * Q) / R
K_max = U_start - U_center
v_max_c = np.sqrt((2 / m) * K_max)

frac_c_c = v_max_c/c

v_max_s = w*x0
frac_c_s = v_max_s/c

print(f'The maximum velocity given by the simple harmonic model is {frac_c_s:.6f}c = {v_max_s:.4e} m/s')
print(f'The maximum velocity given by the non-relativistic model is {frac_c_c:.6f}c = {v_max_c:.4e} m/s')

HTML(ani.to_jshtml())





In [ ]:


Q = 10**(-4)
q = -e
m = m_e
k = 1 / (4 * np.pi * epsilon_0)
R = 0.1
x0 = 1
c = 2.997*(10**8)
v0 = 0

def calculate_t_max_c(x_start, num_cycles=3):
    def U(x):
        return (k * q * Q) / np.sqrt(R**2 + x**2)
    
    E_total = U(x_start)
    
    def time_element(x):
        Kinetic_Energy = E_total - U(x)
        if Kinetic_Energy <= 0: 
            return 0 
        
        velocity = np.sqrt((2/m) * Kinetic_Energy)
        return 1.0 / velocity
        
    quarter_period, error = quad(time_element, 0, x_start * 0.9999)
    period = 4 * quarter_period
    return period * num_cycles

t_max_c = calculate_t_max_c(x0, num_cycles=3)

def diff_eq_c(t,S):
    x, x_dot = S
    dx_dt = x_dot
    dxdot_dt = (k * q * Q * x)/(m*((R**2 + x**2)**(3/2)))
    return [dx_dt, dxdot_dt]

def calculate_t_max_r(x_start, num_cycles=3):
    def U(x):
        return (k * q * Q) / np.sqrt(R**2 + x**2)
    
    E_total = U(x_start)
    
    def time_element(x):
        Kinetic_Energy_r = E_total - U(x)
        if Kinetic_Energy_r <= 0: 
            return np.inf
        
        velocity = np.sqrt(c**2 - ((m**2 * c**6)/(Kinetic_Energy_r + m* c**2)**2))
        return 1.0 / velocity
        
    quarter_period, error = quad(time_element, 0, x_start * 0.9999)
    period = 4 * quarter_period
    return period * num_cycles

t_max_r = calculate_t_max_r(x0, num_cycles=3)

t_max_global = max(t_max_c, t_max_r)
t_eval_global = np.linspace(0, t_max_global, 5000)

sol_c = solve_ivp(diff_eq_c, [0, t_max_global], [x0, v0], t_eval=t_eval_global, method="Radau", rtol=1e-8, atol=1e-10)
x_vals_c = sol_c.y[0]
v_vals_c = sol_c.y[1]
t_vals_c = sol_c.t

def diff_eq_r(t,S):
    x, x_dot = S
    dx_dt = x_dot
    dxdot_dt = (k * q * Q * x / (m * (R**2 + x**2)**(3/2))) * (1 - (x_dot/c)**2)**(3/2)
    return [dx_dt, dxdot_dt]

sol_r = solve_ivp(diff_eq_r, [0, t_max_global], [x0, v0], t_eval=t_eval_global, method="Radau", rtol=1e-8, atol=1e-10)
x_vals_r = sol_r.y[0]
v_vals_r = sol_r.y[1]
t_vals_r = sol_r.t

fig, ((ax, ax2),(ax3,ax4)) = plt.subplots(2, 2, figsize=(12, 10))

ax.set_xlim(0, t_max_global)
ax.set_ylim(-x0 * 1.5, x0 * 1.5)
ax.set_xlabel(r'$t$ (s)')
ax.set_ylabel(r'$x$ (m)')
ax.plot(t_vals_r, x_vals_r, color = 'mediumseagreen', label = 'Relativistic Solution' )
ax.plot(t_vals_c, x_vals_c,  color = 'cornflowerblue', label = 'Non-relativistic Solution' )
ax.legend()
point_rel, = ax.plot([], [], 'go', markersize=8)
point_clas, = ax.plot([], [], 'bo', markersize=8)

ax2.vlines(x=0, ymin=-R, ymax=R, color='black', linewidth=4, label= f"Charged Ring ($Q$ = {Q} C)")
ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlim(-x0*1.5, x0*1.5)
ax2.set_ylim(-R * 1.5, R * 1.5) 
ax2.set_xlabel(r'$x$ (m)')
ax2.set_ylabel(r'$y$ (m)')
particle_rel, = ax2.plot([], [], 'go', markersize=8, label = 'Relativistic Motion')
particle_clas, = ax2.plot([], [], 'bo', markersize=8, label = 'Non-relativistic Motion')
ax2.legend()

U_vals_r = k * q * Q / ((R**2)+(x_vals_r)**2)**0.5

ax3.plot(x_vals_r, U_vals_r, color = 'grey', label = 'Potential Energy')
ax3.set_xlabel(r'$x$ (m)')
ax3.set_ylabel(r'$U$ (J)')
ax3.set_xlim(min(x_vals_r)*1.1, max(x_vals_r)*1.1)
ax3.legend()
point_potential, = ax3.plot([], [], 'ko', markersize=8)

ax4.plot(x_vals_r, v_vals_r, color = 'mediumseagreen', label = 'Relativistic Phase Plot')
ax4.plot(x_vals_c, v_vals_c, color = 'cornflowerblue', label = 'Non-relativistic Phase Plot')
point_phase_r, = ax4.plot([], [], 'go', markersize=8)
point_phase, = ax4.plot([], [], 'bo', markersize=8)

ax4.set_xlabel(r'$x$ (m)')
ax4.set_ylabel(r'$\frac{dx}{dt}$ (m/s)')
ax4.legend()

for axis in [ax, ax2, ax3, ax4]:
    axis.ticklabel_format(style='sci', axis='both', scilimits=(-2, 2))

def init():
    point_rel.set_data([], [])
    point_clas.set_data([], [])
    particle_rel.set_data([], [])
    particle_clas.set_data([], [])
    point_potential.set_data([], [])
    point_phase_r.set_data([], [])
    point_phase.set_data([], [])
    return (point_rel, point_clas, particle_rel, particle_clas, point_potential, point_phase_r, point_phase)

def update(frame):
    point_rel.set_data([t_vals_r[frame]], [x_vals_r[frame]])
    point_clas.set_data([t_vals_c[frame]], [x_vals_c[frame]])
    particle_rel.set_data([x_vals_r[frame]], [0])
    particle_clas.set_data([x_vals_c[frame]], [0])
    point_potential.set_data([x_vals_r[frame]], [U_vals_r[frame]]) 
    point_phase_r.set_data([x_vals_r[frame]], [v_vals_r[frame]])
    point_phase.set_data([x_vals_c[frame]], [v_vals_c[frame]])
    
    return (point_rel, point_clas, particle_rel, particle_clas, point_potential, point_phase_r, point_phase)

step_size = 12 

ani = FuncAnimation(
    fig, 
    update, 
    frames=range(0, len(t_eval_global), step_size),
    init_func=init, 
    blit=True, 
    interval=25
)


A = m**2 * c**6 * R**2 * (R**2 + x0**2)
B = k*q*Q*(R-np.sqrt(R**2 + x0**2)) + m*c**2 * R *np.sqrt(R**2 + x0**2) 
v_max_r = np.sqrt(c**2 - (A/B**2))
frac_c_r = v_max_r/c

v_max_c = np.sqrt(((2*k*q*Q)/m) * ((R - np.sqrt(R**2 + x0**2)) / (R*np.sqrt(R**2 + x0**2))))
frac_c_c = v_max_c/c

print(f'The maximum velocity given by the non-relativistic model is {frac_c_c:.4f}c = {v_max_c:.4e} m/s')
print(f'The maximum velocity given by the relativistic model is {frac_c_r:.4f}c = {v_max_r:.4e} m/s')

delta_t = t_max_r / 3

def integrand(t):
    v_at_t = np.interp(t, t_vals_r, v_vals_r)
    return np.sqrt(1 - (v_at_t / c)**2) 

proper_time, estimated_error = quad(integrand, 0, delta_t)

print(f"Time elapsed in stationary reference frame during one period: {delta_t:.4e} s")
print(f"Time elapsed in electron's reference frame during one period (proper time): {proper_time:.4e} s")


plt.tight_layout()
plt.close()

HTML(ani.to_jshtml())

In [ ]:

Q = 10**(-10)
q = -e
m = m_e
k = 1 / (4 * np.pi * epsilon_0)
R = 0.1
x0 = 1
c = 2.997*(10**8)
v0 = 0


def calculate_t_max_c(x_start, num_cycles=3):
    def U(x):
        return (k * q * Q) / np.sqrt(R**2 + x**2)
    
    E_total = U(x_start)
    
    def time_element(x):
        Kinetic_Energy = E_total - U(x)
        if Kinetic_Energy <= 0: 
            return 0 
        
        velocity = np.sqrt((2/m) * Kinetic_Energy)
        return 1.0 / velocity
        
    quarter_period, error = quad(time_element, 0, x_start * 0.9999)
    period = 4 * quarter_period
    return period * num_cycles

t_max_c = calculate_t_max_c(x0, num_cycles=3)

def diff_eq_c(t,S):
    x, x_dot = S
    dx_dt = x_dot
    dxdot_dt = (k * q * Q * x)/(m*((R**2 + x**2)**(3/2)))
    return [dx_dt, dxdot_dt]

t_eval_global = np.linspace(0, t_max_c, 5000)


sol_c = solve_ivp(diff_eq_c, [0, t_max_c], [x0, v0], t_eval=t_eval_global, method="Radau", rtol=1e-8, atol=1e-10)
x_vals_c = sol_c.y[0]
v_vals_c = sol_c.y[1]
t_vals_c = sol_c.t

scale_factors = [0.2, 0.5, 0.8, 1.3, 1.6, 1.9]
phase_conditions = [x0 * f for f in scale_factors]

max_plot_x = max([x0] + phase_conditions)


fig, ((ax, ax2),(ax3,ax4)) = plt.subplots(2, 2, figsize=(12, 10))

ax.set_xlim(0, t_max_c)
ax.set_ylim(-x0 * 1.5, x0 * 1.5)
ax.set_xlabel(r'$t$ (s)')
ax.set_ylabel(r'$x$ (m)')
ax.plot(t_vals_c, x_vals_c,  color='cornflowerblue', label='Non-relativistic Solution')
ax.legend()
point_clas, = ax.plot([], [], 'bo', markersize=8)

ax2.vlines(x=0, ymin=-R, ymax=R, color='black', linewidth=4, label= f"Charged Ring ($Q$ = {Q} C)")
ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlim(-x0*1.5, x0*1.5)
ax2.set_ylim(-R * 1.5, R * 1.5) 
ax2.set_xlabel(r'$x$ (m)')
ax2.set_ylabel(r'$y$ (m)')
particle_clas, = ax2.plot([], [], 'bo', markersize=8, label='Non-relativistic Motion')
ax2.legend()

U_vals_c = k * q * Q / ((R**2) + (x_vals_c)**2)**0.5
ax3.plot(x_vals_c, U_vals_c, color='grey', label='Potential Energy')
ax3.set_xlabel(r'$x$ (m)')
ax3.set_ylabel(r'$U$ (J)')
ax3.set_xlim(min(x_vals_c)*1.1, max(x_vals_c)*1.1)
ax3.legend()
point_potential, = ax3.plot([], [], 'ko', markersize=8)


for x_init in phase_conditions:

    t_end = calculate_t_max_c(x_init, num_cycles=2)
    t_eval_bg = np.linspace(0, t_end, 1500)
    sol_bg = solve_ivp(diff_eq_c, [0, t_end], [x_init, 0], t_eval=t_eval_bg, method="Radau", rtol=1e-6, atol=1e-8)
    

    ax4.plot(sol_bg.y[0], sol_bg.y[1], color='lightsteelblue', alpha=0.6, linewidth=1, zorder=1)


ax4.plot(x_vals_c, v_vals_c, color='cornflowerblue', linewidth=2.5, label='Main Trajectory', zorder=2)
point_phase, = ax4.plot([], [], 'bo', markersize=8, zorder=3)

ax4.set_xlabel(r'$x$ (m)')
ax4.set_ylabel(r'$\frac{dx}{dt}$ (m/s)')
ax4.legend()

for axis in [ax, ax2, ax3, ax4]:
    axis.ticklabel_format(style='sci', axis='both', scilimits=(-2, 2))


def init():
    point_clas.set_data([], [])
    particle_clas.set_data([], [])
    point_potential.set_data([], [])
    point_phase.set_data([], [])
    return (point_clas, particle_clas, point_potential, point_phase)

def update(frame):
    point_clas.set_data([t_vals_c[frame]], [x_vals_c[frame]])
    particle_clas.set_data([x_vals_c[frame]], [0])
    point_potential.set_data([x_vals_c[frame]], [U_vals_c[frame]]) 
    point_phase.set_data([x_vals_c[frame]], [v_vals_c[frame]])
    
    return (point_clas, particle_clas, point_potential, point_phase)

step_size = 12 

ani = FuncAnimation(
    fig, 
    update, 
    frames=range(0, len(t_eval_global), step_size),
    init_func=init, 
    blit=True, 
    interval=25
)

v_max_c = np.sqrt(((2*k*q*Q)/m) * ((R - np.sqrt(R**2 + x0**2)) / (R*np.sqrt(R**2 + x0**2))))
frac_c_c = v_max_c/c

print(f'The maximum velocity given by the non-relativistic model is {frac_c_c:.4f}c = {v_max_c:.4e} m/s')

plt.tight_layout()
plt.close()

HTML(ani.to_jshtml())

In [ ]:


Q = 10**(-4)
q = -e
m = m_e
k = 1 / (4 * np.pi * epsilon_0)
R = 0.1
x0 = 1.9
c = 2.997*(10**8)
v0 = 0

def calculate_t_max_r(x_start, num_cycles=3):
    def U(x):
        return (k * q * Q) / np.sqrt(R**2 + x**2)
    
    E_total = U(x_start)
    
    def time_element(x):
        Kinetic_Energy_r = E_total - U(x)
        if Kinetic_Energy_r <= 0: 
            return np.inf
        
        velocity = np.sqrt(c**2 - ((m**2 * c**6)/(Kinetic_Energy_r + m* c**2)**2))
        return 1.0 / velocity
        
    quarter_period, error = quad(time_element, 0, x_start * 0.9999)
    period = 4 * quarter_period
    return period * num_cycles

t_max_r = calculate_t_max_r(x0, num_cycles=3)
t_eval_global = np.linspace(0, t_max_r, 5000)

def diff_eq_r(t,S):
    x, x_dot = S
    dx_dt = x_dot
    dxdot_dt = (k * q * Q * x / (m * (R**2 + x**2)**(3/2))) * (1 - (x_dot/c)**2)**(3/2)
    return [dx_dt, dxdot_dt]


sol_r = solve_ivp(diff_eq_r, [0, t_max_r], [x0, v0], t_eval=t_eval_global, method="Radau", rtol=1e-8, atol=1e-10)
x_vals_r = sol_r.y[0]
v_vals_r = sol_r.y[1]
t_vals_r = sol_r.t


scale_factors = [0.2, 0.5, 0.8, 1.3, 1.6, 1.9]
phase_conditions = [x0 * f for f in scale_factors]
max_plot_x = max([x0] + phase_conditions)

layout = [
    ['ax2', 'ax2'],  
    ['ax',  'ax4'],  
    ['ax3', 'ax5']  
]


fig, axes = plt.subplot_mosaic(layout, figsize=(14, 12))


ax = axes['ax']
ax2 = axes['ax2']
ax3 = axes['ax3']
ax4 = axes['ax4']
ax5 = axes['ax5']


ax.set_xlim(0, t_max_r)
ax.set_ylim(-x0 * 1.5, x0 * 1.5)
ax.set_xlabel(r'$t$ (s)')
ax.set_ylabel(r'$x$ (m)')
ax.plot(t_vals_r, x_vals_r, color='mediumseagreen', label='Relativistic Solution')
ax.legend()
point_rel, = ax.plot([], [], 'go', markersize=8)


ax2.vlines(x=0, ymin=-R, ymax=R, color='black', linewidth=4, label= f"Charged Ring ($Q$ = {Q} C)")
ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlim(-x0*1.5, x0*1.5)
ax2.set_ylim(-R * 1.5, R * 1.5) 
ax2.set_xlabel(r'$x$ (m)')
ax2.set_ylabel(r'$y$ (m)')
particle_rel, = ax2.plot([], [], 'go', markersize=8, label='Relativistic Motion')
ax2.legend()


U_vals_r = k * q * Q / ((R**2) + (x_vals_r)**2)**0.5
ax3.plot(x_vals_r, U_vals_r, color='grey', label='Potential Energy')
ax3.set_xlabel(r'$x$ (m)')
ax3.set_ylabel(r'$U$ (J)')
ax3.set_xlim(min(x_vals_r)*1.1, max(x_vals_r)*1.1)
ax3.legend()
point_potential, = ax3.plot([], [], 'ko', markersize=8)


for x_init in phase_conditions:
    t_end = calculate_t_max_r(x_init, num_cycles=2)
    t_eval_bg = np.linspace(0, t_end, 1500)
    sol_bg = solve_ivp(diff_eq_r, [0, t_end], [x_init, 0], t_eval=t_eval_bg, method="Radau", rtol=1e-6, atol=1e-8)
    ax4.plot(sol_bg.y[0], sol_bg.y[1], color='lightsteelblue', alpha=0.6, linewidth=1, zorder=1)

ax4.plot(x_vals_r, v_vals_r, color='mediumaquamarine', linewidth=2.5, label='Main Trajectory', zorder=2)
point_phase_r, = ax4.plot([], [], 'go', markersize=8, zorder=3)
ax4.set_xlabel(r'$x$ (m)')
ax4.set_ylabel(r'$\frac{dx}{dt}$ (m/s)')
ax4.legend()

inverse_gamma_vals = np.sqrt(1 - (v_vals_r / c)**2)
ax5.plot(t_vals_r, inverse_gamma_vals, color='mediumseagreen')
ax5.set_xlabel(r'$t$ (s)')
ax5.set_ylabel(r'$\frac{d\tau}{dt} = \frac{1}{\gamma}$')
point_gamma, = ax5.plot([], [], 'go', markersize=8)


for axis in [ax, ax2, ax3, ax4, ax5]:
    axis.ticklabel_format(style='sci', axis='both', scilimits=(-2, 2))


def init():
    point_rel.set_data([], [])
    particle_rel.set_data([], [])
    point_potential.set_data([], [])
    point_phase_r.set_data([], [])
    point_gamma.set_data([], [])
    return (point_rel, particle_rel, point_potential, point_phase_r, point_gamma)

def update(frame):
    point_rel.set_data([t_vals_r[frame]], [x_vals_r[frame]])
    particle_rel.set_data([x_vals_r[frame]], [0])
    point_potential.set_data([x_vals_r[frame]], [U_vals_r[frame]]) 
    point_phase_r.set_data([x_vals_r[frame]], [v_vals_r[frame]])
    point_gamma.set_data([t_vals_r[frame]], [inverse_gamma_vals[frame]])
    
    return (point_rel, particle_rel, point_potential, point_phase_r, point_gamma)

step_size = 12 

ani = FuncAnimation(
    fig, 
    update, 
    frames=range(0, len(t_eval_global), step_size),
    init_func=init, 
    blit=True, 
    interval=25
)


A = m**2 * c**6 * R**2 * (R**2 + x0**2)
B = k*q*Q*(R-np.sqrt(R**2 + x0**2)) + m*c**2 * R *np.sqrt(R**2 + x0**2) 
v_max_r = np.sqrt(c**2 - (A/B**2))
frac_c_r = v_max_r/c

print(f'The maximum velocity given by the relativistic model is {frac_c_r:.4f}c = {v_max_r:.4e} m/s')

delta_t = t_max_r / 3

def integrand(t):
    v_at_t = np.interp(t, t_vals_r, v_vals_r)
    return np.sqrt(1 - (v_at_t / c)**2) 

proper_time, estimated_error = quad(integrand, 0, delta_t)

print(f"Time elapsed in stationary reference frame during one period: {delta_t:.4e} s")
print(f"Time elapsed in electron's reference frame during one period (proper time): {proper_time:.4e} s")

plt.tight_layout()
plt.close()

HTML(ani.to_jshtml())

